In [1]:
import os
import math
import cv2
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [2]:
# ============================================================
# Dataset: Document Shadow Removal (Jung, CSV)
# ============================================================

class DocumentShadowDataset(Dataset):
    """
    Dataset dokumen dengan bayangan.
    CSV format (contoh):

        Unnamed: 0, img, gt, B, G, R
        0, ./dataset/Jung/train/img/rect_31.jpg, ./dataset/Jung/train/gt/rect_31.jpg, ...

    Yang dipakai cuma kolom:
        - img : path shadow
        - gt  : path ground truth

    Kolom B, G, R diabaikan (model sudah punya BE-Net).
    """
    def __init__(self, csv_path, img_size=512, augment=False):
        super().__init__()
        self.df = pd.read_csv(csv_path)

        if not {"img", "gt"}.issubset(self.df.columns):
            raise ValueError("CSV harus memiliki kolom 'img' dan 'gt'")

        self.img_paths = self.df["img"].tolist()
        self.gt_paths  = self.df["gt"].tolist()

        self.img_size = img_size
        self.augment = augment
        self.to_tensor = transforms.ToTensor()

    def __len__(self):
        return len(self.img_paths)

    def _load_image(self, full_path):
        img = cv2.imread(full_path, cv2.IMREAD_COLOR)
        if img is None:
            raise FileNotFoundError(f"Image not found: {full_path}")

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (self.img_size, self.img_size), interpolation=cv2.INTER_AREA)
        img = img.astype(np.float32) / 255.0
        return img

    def __getitem__(self, idx):
        shadow_path = self.img_paths[idx]
        gt_path     = self.gt_paths[idx]

        shadow = self._load_image(shadow_path)
        gt     = self._load_image(gt_path)

        if self.augment:
            if np.random.rand() < 0.5:
                shadow = np.fliplr(shadow).copy()
                gt     = np.fliplr(gt).copy()

        shadow = self.to_tensor(shadow)   # [3,H,W]
        gt     = self.to_tensor(gt)       # [3,H,W]

        return shadow, gt

In [3]:
# ============================================================
# U-Net building blocks
# ============================================================

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class Down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_ch, out_ch)
        )

    def forward(self, x):
        return self.net(x)


class Up(nn.Module):
    def __init__(self, in_ch, out_ch, bilinear=True):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
            self.conv = DoubleConv(in_ch, out_ch)
        else:
            self.up = nn.ConvTranspose2d(in_ch // 2, in_ch // 2, 2, stride=2)
            self.conv = DoubleConv(in_ch, out_ch)

    def forward(self, x1, x2):
        x1 = self.up(x1)

        # padding biar size sama
        diffY = x2.size(2) - x1.size(2)
        diffX = x2.size(3) - x1.size(3)
        x1 = F.pad(
            x1,
            [diffX // 2, diffX - diffX // 2,
             diffY // 2, diffY - diffY // 2]
        )

        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)


class OutConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, 1)

    def forward(self, x):
        return self.conv(x)


In [4]:
# ============================================================
# BE-Net: Background Estimation + Attention
# ============================================================

class BENet(nn.Module):
    """
    Input  : dokumen shadow [B,3,H,W]
    Output : bg_color [B,3]     (average background doc)
             att_map  [B,1,H,W] (shadow attention)
    """
    def __init__(self, in_ch=3, base_ch=16):
        super().__init__()
        self.enc1 = DoubleConv(in_ch, base_ch)
        self.enc2 = Down(base_ch, base_ch * 2)
        self.enc3 = Down(base_ch * 2, base_ch * 4)

        self.att_head = nn.Conv2d(base_ch * 4, 1, 1)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc_bg = nn.Linear(base_ch * 4, 3)

    def forward(self, x):
        x1 = self.enc1(x)
        x2 = self.enc2(x1)
        x3 = self.enc3(x2)

        # attention di feature map deep
        att = torch.sigmoid(self.att_head(x3))
        att = F.interpolate(att, size=x.shape[2:], mode="bilinear", align_corners=False)

        # background color global
        g = self.gap(x3).view(x3.size(0), -1)
        bg = torch.sigmoid(self.fc_bg(g))  # [B,3] / 0-1

        return bg, att

In [5]:
# ============================================================
# SR-Net: Shadow Removal (U-Net style)
# ============================================================

class SRNet(nn.Module):
    """
    Input : concat(shadow_img[3], bg_map[3], att[1]) -> [B,7,H,W]
    Output: dokumen bebas bayangan [B,3,H,W]
    """
    def __init__(self, in_ch=7, out_ch=3, base_ch=32):
        super().__init__()
        self.inc   = DoubleConv(in_ch, base_ch)
        self.down1 = Down(base_ch, base_ch * 2)
        self.down2 = Down(base_ch * 2, base_ch * 4)
        self.down3 = Down(base_ch * 4, base_ch * 8)

        self.up1   = Up(base_ch * 8 + base_ch * 4, base_ch * 4)
        self.up2   = Up(base_ch * 4 + base_ch * 2, base_ch * 2)
        self.up3   = Up(base_ch * 2 + base_ch, base_ch)

        self.outc  = OutConv(base_ch, out_ch)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)

        x = self.up1(x4, x3)
        x = self.up2(x,  x2)
        x = self.up3(x,  x1)

        out = self.outc(x)
        out = torch.sigmoid(out)
        return out


In [6]:
# ============================================================
# Full BEDSR-like model
# ============================================================

class BEDSRNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.benet = BENet()
        self.srnet = SRNet()

    def forward(self, shadow_img):
        """
        shadow_img: [B,3,H,W]
        return:
          pred_img : [B,3,H,W]
          bg_color : [B,3]
          att_map  : [B,1,H,W]
        """
        bg_color, att_map = self.benet(shadow_img)

        B, _, H, W = shadow_img.shape
        bg_map = bg_color.view(B, 3, 1, 1).expand(-1, -1, H, W)

        x_sr = torch.cat([shadow_img, bg_map, att_map], dim=1)  # [B,7,H,W]
        pred = self.srnet(x_sr)

        return pred, bg_color, att_map


In [7]:
# ============================================================
# Metrics: PSNR & SSIM
# ============================================================

def calc_psnr(pred, target, eps=1e-8):
    """
    pred, target: [B,3,H,W], 0-1
    """
    mse = F.mse_loss(pred, target, reduction="mean")
    if mse.item() <= 0:
        return 99.0
    psnr = 10 * math.log10(1.0 / (mse.item() + eps))
    return psnr


def _gaussian_window(window_size=11, sigma=1.5, channels=3, device="cpu"):
    coords = torch.arange(window_size, dtype=torch.float32, device=device) - window_size // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    kernel_1d = g.view(1, 1, 1, -1)          # [1,1,1,W]
    kernel_2d = kernel_1d.transpose(2, 3) @ kernel_1d  # [1,1,H,W]
    kernel_2d = kernel_2d / kernel_2d.sum()
    kernel_2d = kernel_2d.expand(channels, 1, window_size, window_size)
    return kernel_2d


def calc_ssim(pred, target, window_size=11, sigma=1.5, C1=0.01**2, C2=0.03**2):
    """
    pred, target: [B,3,H,W], 0-1
    return: average SSIM (float)
    """
    pred   = torch.clamp(pred,   0.0, 1.0)
    target = torch.clamp(target, 0.0, 1.0)

    B, C, H, W = pred.shape
    device = pred.device

    window = _gaussian_window(window_size, sigma, C, device=device)
    padding = window_size // 2

    mu_x = F.conv2d(pred,   window, padding=padding, groups=C)
    mu_y = F.conv2d(target, window, padding=padding, groups=C)

    mu_x2 = mu_x * mu_x
    mu_y2 = mu_y * mu_y
    mu_xy = mu_x * mu_y

    sigma_x2 = F.conv2d(pred * pred,     window, padding=padding, groups=C) - mu_x2
    sigma_y2 = F.conv2d(target * target, window, padding=padding, groups=C) - mu_y2
    sigma_xy = F.conv2d(pred * target,   window, padding=padding, groups=C) - mu_xy

    ssim_map = ((2 * mu_xy + C1) * (2 * sigma_xy + C2)) / \
               ((mu_x2 + mu_y2 + C1) * (sigma_x2 + sigma_y2 + C2))

    return ssim_map.mean().item()

In [8]:
# ============================================================
# Train / Val / Test
# ============================================================

from torch.cuda.amp import autocast, GradScaler
scaler = GradScaler()


def train_one_epoch(model, loader, optimizer, device, epoch, num_epochs):
    model.train()
    total_loss = 0.0
    total_batches = len(loader)

    print(f"\n===== Epoch {epoch} / {num_epochs} =====")
    print(f"Total batches: {total_batches}")

    for batch_idx, (shadow, gt) in enumerate(loader, start=1):
        shadow = shadow.to(device)
        gt     = gt.to(device)

        with autocast():
            pred, bg, att = model(shadow)
            loss = F.l1_loss(pred, gt)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()


        total_loss += loss.item() * shadow.size(0)

        # progress bar ----------
        progress = batch_idx / total_batches
        bar_len = 30
        filled = int(progress * bar_len)
        bar = "=" * filled + "-" * (bar_len - filled)

        print(
            f"\r[Epoch {epoch}] [{bar}] "
            f"{batch_idx}/{total_batches} "
            f"loss={loss.item():.4f}",
            end=""
        )

    print("")  # biar rapi
    return total_loss / len(loader.dataset)



C:\Users\Lenovo Legion\AppData\Local\Temp\ipykernel_22404\665620206.py:6: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [9]:
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss = 0.0
    total_psnr = 0.0
    total_ssim = 0.0
    n = 0

    for shadow, gt in loader:
        shadow = shadow.to(device)
        gt     = gt.to(device)

        pred, bg, att = model(shadow)
        loss = F.l1_loss(pred, gt)

        psnr = calc_psnr(pred, gt)
        ssim = calc_ssim(pred, gt)

        bsz = shadow.size(0)
        total_loss += loss.item() * bsz
        total_psnr += psnr * bsz
        total_ssim += ssim * bsz
        n += bsz

    return (
        total_loss / n,
        total_psnr / n,
        total_ssim / n,
    )

In [10]:
@torch.no_grad()
def run_test_and_save(model, loader, device, save_root, csv_path):
    """
    Run inference di test set dan simpan hasil shadow removal.
    Nama file output diambil dari nama file 'img' + suffix _sr.
    """
    os.makedirs(save_root, exist_ok=True)
    model.eval()

    df = pd.read_csv(csv_path)
    img_list = df["img"].tolist()

    idx_global = 0
    for shadow, gt in loader:
        shadow = shadow.to(device)
        pred, bg, att = model(shadow)

        for i in range(pred.size(0)):
            img = pred[i].cpu().permute(1, 2, 0).numpy()
            img = (img * 255.0).clip(0, 255).astype(np.uint8)
            img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

            img_rel = img_list[idx_global]
            base_name = os.path.basename(img_rel)
            out_name = os.path.splitext(base_name)[0] + "_sr.png"

            cv2.imwrite(os.path.join(save_root, out_name), img)
            idx_global += 1

In [11]:
# ============================================================
# Main
# ============================================================

from torch.cuda.amp import autocast, GradScaler
scaler = GradScaler()

def main():
    # path CSV
    csv_root  = "./csv/Jung"
    train_csv = os.path.join(csv_root, "train.csv")
    val_csv   = os.path.join(csv_root, "val.csv")
    test_csv  = os.path.join(csv_root, "test.csv")

    img_size  = 512
    batch_size  = 4      # bisa dinaikkan kalau GPU kuat
    num_workers = 0
    num_epochs  = 100
    lr          = 1e-4

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)

    train_ds = DocumentShadowDataset(train_csv, img_size, augment=True)
    val_ds   = DocumentShadowDataset(val_csv,   img_size, augment=False)
    test_ds  = DocumentShadowDataset(test_csv,  img_size, augment=False)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, pin_memory=True)

    model = BEDSRNet().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # skor gabungan PSNR + SSIM*10 buat milih model terbaik
    best_score = 0.0

    for epoch in range(1, num_epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, device, epoch, num_epochs)
        val_loss, val_psnr, val_ssim = evaluate(model, val_loader, device)

        print(
            f"[Epoch {epoch:03d}] "
            f"train_loss={train_loss:.4f} | "
            f"val_loss={val_loss:.4f} | "
            f"val_psnr={val_psnr:.2f} dB | "
            f"val_ssim={val_ssim:.4f}"
        )

        score = val_psnr + (val_ssim * 10.0)

        if score > best_score:
            best_score = score
            os.makedirs("checkpoints", exist_ok=True)
            ckpt_path = os.path.join("checkpoints", "bedsrnet_jung_best.pth")
            torch.save(model.state_dict(), ckpt_path)
            print(
                f"  -> Save best model: {ckpt_path} "
                f"(PSNR={val_psnr:.2f} dB, SSIM={val_ssim:.4f})"
            )

    # setelah training, run test & simpan hasil
    print("Run inference di test set...")
    run_test_and_save(
        model,
        test_loader,
        device,
        save_root="./results_jung",
        csv_path=test_csv
    )
    print("Done. Results saved in ./results_jung")


if __name__ == "__main__":
    main()

C:\Users\Lenovo Legion\AppData\Local\Temp\ipykernel_22404\2624801219.py:6: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
C:\Users\Lenovo Legion\AppData\Local\Temp\ipykernel_22404\665620206.py:21: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Device: cuda

===== Epoch 1 / 100 =====
Total batches: 15
[Epoch 1] [==============================] 15/15 loss=0.2540
[Epoch 001] train_loss=0.2612 | val_loss=0.2954 | val_psnr=10.22 dB | val_ssim=0.5920
  -> Save best model: checkpoints\bedsrnet_jung_best.pth (PSNR=10.22 dB, SSIM=0.5920)

===== Epoch 2 / 100 =====
Total batches: 15
[Epoch 2] [==============================] 15/15 loss=0.2271
[Epoch 002] train_loss=0.2346 | val_loss=0.2356 | val_psnr=11.98 dB | val_ssim=0.6214
  -> Save best model: checkpoints\bedsrnet_jung_best.pth (PSNR=11.98 dB, SSIM=0.6214)

===== Epoch 3 / 100 =====
Total batches: 15
[Epoch 3] [==============================] 15/15 loss=0.2342
[Epoch 003] train_loss=0.2197 | val_loss=0.1804 | val_psnr=14.01 dB | val_ssim=0.6720
  -> Save best model: checkpoints\bedsrnet_jung_best.pth (PSNR=14.01 dB, SSIM=0.6720)

===== Epoch 4 / 100 =====
Total batches: 15
[Epoch 4] [==============================] 15/15 loss=0.2195
[Epoch 004] train_loss=0.2124 | val_loss=0.1801